In [29]:
### Semantic Chunking
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

In [30]:
# Initilize the model

model = SentenceTransformer('all-MiniLM-L6-v2')

## Sample text
text="""
LangChain is a framework for building applications with LLMs.
Langchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.
You can create chains, agents, memory, and retrievers.
The Eiffel Tower is located in Paris.
France is a popular tourist destination.
"""


## Step1: Split into sentences

sentences = [s.strip() for s in text.split("\n") if s.strip()]

### Step2 : Embeded each sentence

embeddings = model.encode(sentences)


# Step3" Initialize paramters this is the threshold decide by product like how much should be thrshold
# so that same sentence/vector embedding at same palce

threshold = 0.7 # control chunk tightness
chunks = []
current_chunk=[sentences[0]]



# Step4 Sematic grouping based on threshold

for i in range(1, len(sentences)):
    sim = cosine_similarity(
        [embeddings[i - 1]],
        [embeddings[i]]
    )[0][0]

    if sim >= threshold:
        current_chunk.append(sentences[i])
    else:
        chunks.append(" ".join(current_chunk))
        current_chunk=[sentences[i]]


# Append last chunk

chunks.append(" ".join(current_chunk))


# Output the chunks
print("\n📌 Semantic Chunks:")
for idx, chunk in enumerate(chunks):
    print(f"\nChunk {idx+1}:\n{chunk}")


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5119.61it/s]



📌 Semantic Chunks:

Chunk 1:
LangChain is a framework for building applications with LLMs. Langchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.

Chunk 2:
You can create chains, agents, memory, and retrievers.

Chunk 3:
The Eiffel Tower is located in Paris.

Chunk 4:
France is a popular tourist destination.


### RAG Pipeline Modular Coding

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from langchain.chat_models import init_chat_model
from langchain_core.runnables import RunnableLambda, RunnableMap
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
import os
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langchain_ollama import ChatOllama
from langchain_astradb import AstraDBVectorStore


embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

embeddings


ASTRA_DB_API_ENDPOINT = ""
ASTRA_DB_APPLICATION_TOKEN = ""



vector_store = AstraDBVectorStore(
    collection_name="rag_sematic_chunking",
    embedding=embeddings,
    api_endpoint=ASTRA_DB_API_ENDPOINT,
    token=ASTRA_DB_APPLICATION_TOKEN,
    namespace=None,
)
vector_store

llm = ChatOllama(
    model="qwen2.5:3b",
    base_url="http://localhost:11434"
)




Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2819.91it/s]


In [32]:
### Custom Semantic Chunker With Threshold

class ThresholdSematicChunker:
    def __init__(self,threshold,model_name="all-MiniLM-L6-v2"):
        self.model=SentenceTransformer(model_name)
        self.threshold=threshold 

    def split(self, text: str):
        sentences = [s.strip() for s in text.split('.') if s.strip()]
        embeddings = self.model.encode(sentences)    #The primary difference is their ecosystem: encode is the native method used by Hugging Face’s SentenceTransformers library, while embed_query is a method used by orchestration frameworks like LangChain to interface with Hugging Face models.
        chunks = []                                  # Sentence transformer is huggin face library thats why encode
        current_chunk = [sentences[0]]

        for i in range(1, len(sentences)):
            sim = cosine_similarity([embeddings[i - 1]], [embeddings[i]])[0][0]
            if sim >= self.threshold:
                current_chunk.append(sentences[i])
            else:
                chunks.append(". ".join(current_chunk) + ".")
                current_chunk = [sentences[i]]

        chunks.append(". ".join(current_chunk) + ".")
        return chunks
    
    def split_documents(self,docs):
        result=[]
        for doc in docs:
            for chunk in self.split(doc.page_content):
                result.append(Document(page_content=chunk, metadata=doc.metadata))

        return result

    

In [38]:
# Sample text
sample_text = """
LangChain is a framework for building applications with LLMs.
Langchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.
You can create chains, agents, memory, and retrievers.
The Eiffel Tower is located in Paris.
France is a popular tourist destination.
"""

doc = Document(page_content=sample_text)
doc


### Chunking
chunker=ThresholdSematicChunker(0.7)
chunks=chunker.split_documents([doc])
chunks

vector_store.add_documents(documents=chunks)

docs = vector_store.similarity_search(
    "What is LangChain used for?",
    k=3
)

print("Number of docs:", len(docs))

for doc in docs:
    print("\n----------------")
    print(doc.page_content)
    print(doc.metadata)





Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4010.97it/s]


Number of docs: 3

----------------
LangChain is a framework for building applications with LLMs. Langchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.
{}

----------------
LangChain is a framework for building applications with LLMs. Langchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.
{}

----------------
LangChain is a framework for building applications with LLMs. Langchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.
{}


In [36]:
retriever=vector_store.as_retriever()


template = """
Answer the question using ONLY the information provided in the context.

If the answer cannot be found in the context, say:
"I don't know based on the provided context."

Do not use your own knowledge.

Context:
{context}

Question:
{question}
"""

prompt = PromptTemplate.from_template(template)

In [ ]:


### LCEL Chain With retrieval

rag_chain=(
    RunnableMap(
        {
        "context": lambda x: retriever.invoke(x["question"]),
        "question": lambda x: x["question"],  
        }
    )
    | prompt
    | llm
    | StrOutputParser()
)

# --- 8. Run Query ---
query = {"question": "What is LangChain used for?"}
result = rag_chain.invoke(query)
print(prompt)
print(result)

input_variables=['context', 'question'] input_types={} partial_variables={} template='\nAnswer the question using ONLY the information provided in the context.\n\nIf the answer cannot be found in the context, say:\n"I don\'t know based on the provided context."\n\nDo not use your own knowledge.\n\nContext:\n{context}\n\nQuestion:\n{question}\n'
LangChain is used for building applications with LLMs by providing modular abstractions to combine LLMs with tools like OpenAI and Pinecone.


### Semantic chunker With Langchain (internal function)

In [41]:
from langchain_experimental.text_splitter import SemanticChunker
from langchain.document_loaders import TextLoader

## Load the documents
loader=TextLoader("langchain_intro.txt")
docs=loader.load()

## Create the semantic chunker
chunker=SemanticChunker(embeddings)

## Split the documents
chunks=chunker.split_documents(docs)

## Result

for i,chunk in enumerate(chunks):
    print(f"\n chunk {i+1}:\n{chunk.page_content}")

ModuleNotFoundError: No module named 'langchain.document_loaders'

 chunk 1:
LangChain is a framework for building applications with LLMs. Langchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.

 chunk 2:
You can create chains, agents, memory, and retrievers. The Eiffel Tower is located in Paris. France is a popular tourist destination.